# 02 — Capacidade de Consumo e Dinamismo Econômico

**Objetivo**: analisar renda, PIB, volume Pix e dinamismo financeiro dos municípios.

**Inputs**: `data/processed/trusted_municipios_eda.parquet`, `raw_bcb_pix_transacoes`.

**Outputs**: figuras e tabela de estatísticas por estrato populacional.

In [ ]:
# Imports
import logging

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.config import PROCESSED_DATA_DIR
from src.utils.bigquery import read_table_to_dataframe
from src.utils.eda import plot_boxplot_by_group, plot_distribution, save_figure, save_json

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100

In [ ]:
# Leitura
df = pd.read_parquet(PROCESSED_DATA_DIR / "trusted_municipios_eda.parquet")
df_raw_pix = read_table_to_dataframe("raw_bcb_pix_transacoes")
logger.info("trusted: %s, raw_pix: %s", df.shape, df_raw_pix.shape)

## 1. Distribuição de renda e PIB

In [ ]:
for col in ["rendimento_domiciliar_per_capita", "pib_per_capita"]:
    plot_distribution(df, col, log_scale=True, filename=f"02_dist_log_{col}.png")
    plt.show()

## 2. Pix vs PIB per capita

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(
    data=df, x="pib_per_capita", y="pix_per_capita_12m", hue="nome_regiao", alpha=0.6, ax=ax
)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_title("Pix per capita vs PIB per capita")
save_figure(fig, "02_pix_vs_pib.png")
plt.show()

## 3. Sazonalidade e evolução do Pix

In [ ]:
df_raw_pix["ano_mes"] = pd.to_datetime(
    df_raw_pix["AnoMes"].astype(str), format="%Y%m"
)

serie = (
    df_raw_pix.groupby("ano_mes")
    .agg(
        VL_PagadorPF=("VL_PagadorPF", "sum"),
        VL_PagadorPJ=("VL_PagadorPJ", "sum"),
    )
    .reset_index()
    .sort_values("ano_mes")
)

fig, ax = plt.subplots(figsize=(12, 6))
serie.plot(x="ano_mes", y=["VL_PagadorPF", "VL_PagadorPJ"], ax=ax)
ax.set_title("Evolução do volume Pix (PF vs PJ)")
ax.set_ylabel("Volume (R$)")
ax.tick_params(axis="x", rotation=45)
save_figure(fig, "02_serie_pix.png")
plt.show()

## 4. Estatísticas por estrato populacional

In [ ]:
stats = (
    df.groupby("estrato_populacional")
    .agg(
        rendimento_domiciliar_per_capita=("rendimento_domiciliar_per_capita", "median"),
        pib_per_capita=("pib_per_capita", "median"),
        pix_per_capita_12m=("pix_per_capita_12m", "median"),
    )
    .round(2)
)
display(stats)

## 5. Boxplots por estrato

In [ ]:
for col in ["rendimento_domiciliar_per_capita", "pib_per_capita", "pix_per_capita_12m"]:
    plot_boxplot_by_group(
        df, col, "estrato_populacional", filename=f"02_boxplot_{col}_estrato.png"
    )
    plt.show()

## 6. Municípios com alta adoção Pix e baixa renda

In [ ]:
mediana_pix = df["pix_per_capita_12m"].median()
mediana_renda = df["rendimento_domiciliar_per_capita"].median()

df_oportunidade = df[
    (df["pix_per_capita_12m"] > mediana_pix)
    & (df["rendimento_domiciliar_per_capita"] < mediana_renda)
]

logger.info("Municípios com alto Pix e baixa renda: %d", len(df_oportunidade))
display(
    df_oportunidade[["nome_municipio", "sigla_uf", "pix_per_capita_12m", "rendimento_domiciliar_per_capita"]]
    .sort_values("pix_per_capita_12m", ascending=False)
    .head(10)
)

In [ ]:
report = {
    "estatisticas_por_estrato": stats.to_dict(),
    "municipios_alto_pix_baixa_renda": int(len(df_oportunidade)),
}
save_json(report, "02_economia_dinamismo.json")